# Build 9 - Final Submission Strategy

**Objective:** make the final competition submission decision using the
validated evidence accumulated across Builds 1-8, actual Kaggle
constraints, model diversity, public-LB behavior, and private-LB risk.
Decision and competition-close preparation only -- no new feature
engineering, hyperparameter tuning, model-family benchmarking, blending,
stacking, or exploratory modeling, unless a genuine integrity problem is
found in an existing candidate (none was).

Entering this build: E010 (tuned XGBoost) is the settled primary
candidate (CV 0.96499, public LB 0.96653). The one open question is
whether E008 (CatBoost + `screen_residual`) should be retained as a
second final-submission hedge -- it is genuinely diverse from E010 but
has never been submitted to Kaggle.

## 1. Current competition constraints

Verified on the live competition page
(`kaggle.com/competitions/playground-series-s6e8`), checked 2026-08-24.
See `docs/COMPETITION_NOTES.md` for the full record.

In [1]:
constraints = {
    "competition_status": "Open",
    "days_to_go_at_check": 7,
    "final_submission_deadline": "2026-08-31 23:59 UTC",
    "daily_submission_limit": 10,
    "final_submissions_allowed": 2,
    "manual_final_selection_required": True,
    "prize_structure": "Kaggle merchandise only -- no points or medals",
}
for k, v in constraints.items():
    print(f"{k}: {v}")

competition_status: Open
days_to_go_at_check: 7
final_submission_deadline: 2026-08-31 23:59 UTC
daily_submission_limit: 10
final_submissions_allowed: 2
manual_final_selection_required: True
prize_structure: Kaggle merchandise only -- no points or medals


## 2. Candidate evidence

Canonical final-candidate table, reconstructed from
`experiments/experiments.csv` and Build 7/8 diversity artifacts.
Persisted to `outputs/final_submission_candidates.csv`.

In [2]:
import pandas as pd

candidates = pd.read_csv("../outputs/final_submission_candidates.csv")
candidates

,experiment_id,model,feature_set,role,cv_mean,cv_std,public_lb,pearson_vs_e010,spearman_vs_e010,top_decile_disagreement_vs_e010,model_family,candidate_status,selection_reason,primary_risk
0,E010,XGBClassifier (tuned),raw + screen_residual,Primary final submission,0.96499,0.00051,0.96653,1.000000,1.000000,0.000000,XGBoost,Selected,Best CV and best public LB of every experiment...,Entire winning lineage (E004/E006/E010) is sin...
1,E008,CatBoostClassifier,raw + screen_residual,Second final submission (hedge),0.96104,0.00055,PENDING,0.984872,0.984679,0.280501,CatBoost,Selected,Only historical candidate with genuine predict...,Never submitted before this build -- public LB...
2,E006,XGBClassifier (pre-tuned),raw + screen_residual,Historical reference,0.96445,0.00056,0.96608,0.996747,0.997404,0.103606,XGBoost,Rejected as hedge,Build 7 found E006 too redundant with E010 (0....,Not applicable -- not a final candidate.
3,E004,XGBClassifier,raw predictors,Historical reference,0.96382,0.00056,0.96539,NaN,NaN,NaN,XGBoost,Dominated,Superseded by E006/E010 on both CV and public ...,Not applicable -- not a final candidate.
4,E002,CatBoostClassifier,raw predictors,Historical reference,0.96040,0.00051,0.96151,NaN,NaN,NaN,CatBoost,Dominated,Same model family as E008 with a strictly wors...,Not applicable -- not a final candidate.
5,E001,LogisticRegression,raw predictors,Historical reference,0.91149,0.00081,0.91358,NaN,NaN,NaN,Linear,Dominated,"Baseline; ~5.3pt CV AUC gap to E010, too weak ...",Not applicable -- not a final candidate.


## 3. E010 integrity confirmation

E010's committed deliverable (`deliverables/submission_E010_xgb_tuned.csv`)
is cross-checked against the persisted, `cv_mean`-verified
`outputs/test_predictions/E010.csv` artifact (Build 7 regeneration) to
confirm no drift and no dependency on uncommitted local code.

In [3]:
import numpy as np
import sys
sys.path.insert(0, "..")

from src.config import SAMPLE_SUBMISSION_PATH, ID_COLUMN, TARGET_COLUMN, DELIVERABLES_DIR
from src.ensembling import load_test_pred
from src.submission_validation import validate_submission

sample = pd.read_csv(SAMPLE_SUBMISSION_PATH)

e010_sub = pd.read_csv(DELIVERABLES_DIR / "submission_E010_xgb_tuned.csv")
validate_submission(e010_sub, sample)

e010_artifact = load_test_pred("E010", sample[ID_COLUMN])
e010_sub_aligned = (
    e010_sub.set_index(ID_COLUMN).loc[sample[ID_COLUMN], TARGET_COLUMN].reset_index(drop=True)
)
max_diff = float(np.abs(e010_sub_aligned.to_numpy() - e010_artifact.to_numpy()).max())

print("E010 deliverable schema validation: PASSED")
print(f"E010 deliverable rows: {len(e010_sub)}")
print(f"Max abs diff vs persisted test_predictions/E010.csv artifact: {max_diff:.2e}")
assert max_diff < 1e-6, "E010 deliverable diverges from its recorded prediction artifact"
print("Result: no drift -- E010 remains reproducible from committed artifacts.")

E010 deliverable schema validation: PASSED
E010 deliverable rows: 296302
Max abs diff vs persisted test_predictions/E010.csv artifact: 5.00e-09
Result: no drift -- E010 remains reproducible from committed artifacts.


**Conclusion: E010 retained as primary.** No reproducibility or
integrity defect was found. Its CV/LB standing (5/5 folds improved over
E006, tighter CV std, confirmed public LB improvement) was already
established in Builds 6 and 8 and is not reopened here -- per the
build's own rule, a small numeric tuning gain is not grounds to resume
tuning.

## 4. E008 hedge assessment

Diversity evidence from Build 7 (`outputs/ensemble_prediction_correlations.csv`),
individual strength from `experiments/experiments.csv`.

In [4]:
diversity = pd.read_csv("../outputs/ensemble_prediction_correlations.csv")
e010_e008 = diversity[
    ((diversity.model_a == "E010") & (diversity.model_b == "E008"))
    | ((diversity.model_a == "E008") & (diversity.model_b == "E010"))
]
e010_e006 = diversity[
    ((diversity.model_a == "E010") & (diversity.model_b == "E006"))
    | ((diversity.model_a == "E006") & (diversity.model_b == "E010"))
]
print("E010 vs E008 (candidate hedge):")
print(e010_e008.to_string(index=False))
print()
print("E010 vs E006 (rejected hedge, for contrast):")
print(e010_e006.to_string(index=False))

E010 vs E008 (candidate hedge):
model_a model_b  pearson  spearman  mean_abs_diff  top_decile_disagreement  bottom_decile_disagreement
   E010    E008 0.984872  0.984679       0.035746                 0.280501                    0.116045

E010 vs E006 (rejected hedge, for contrast):
model_a model_b  pearson  spearman  mean_abs_diff  top_decile_disagreement  bottom_decile_disagreement
   E010    E006 0.996747  0.997404       0.015132                 0.103606                     0.05628


**Hedge decision framework:**

- *Strength*: E008 CV 0.96104 vs E010's 0.96499 -- a real but modest
  gap (~0.004 AUC).
- *Stability*: E008 CV std 0.00055, within the same tight band as every
  other boosting-model experiment (0.00051-0.00056).
- *Diversity*: E010 vs E008 Pearson 0.985, 28.1% top-decile
  disagreement -- clearly more diverse than E010 vs E006 (0.997 Pearson,
  10.4% top-decile disagreement).
- *Model-family independence*: CatBoost vs E010's XGBoost -- directly
  addresses the model-family-concentration risk factor Build 8 flagged.
- *Feature dependence*: identical accepted feature set (raw predictors +
  `screen_residual`) -- diversifies model family only.
- *Existing LB evidence*: none before this build.

**Decision: retain E008 as the second final submission (hedge).** This
decision is recorded in `docs/DECISIONS.md` *before* E008's public LB
score is observed, per the build's no-leaderboard-chasing rule. E006 is
not reconsidered -- Build 7 already found it too redundant with E010
(0.997 Pearson) to offer any diversification benefit.

## 5. E008 submission generation

E008 is submitted **exactly as already validated** -- no parameter,
feature, seed, categorical handling, fold setup, iteration budget, or
prediction method is changed. The submission file is generated by
loading the existing, `cv_mean`-verified `outputs/test_predictions/E008.csv`
artifact (Build 7 regeneration, reconfirmed against E008's recorded
`cv_mean` of 0.96104 before being trusted) -- no retraining.

In [5]:
e008_preds = load_test_pred("E008", sample[ID_COLUMN])
e008_submission = pd.DataFrame(
    {ID_COLUMN: sample[ID_COLUMN], TARGET_COLUMN: e008_preds.to_numpy()}
)
validate_submission(e008_submission, sample)

out_path = DELIVERABLES_DIR / "submission_E008_catboost_screen_residual.csv"
e008_submission.to_csv(out_path, index=False)

print("E008 submission schema validation: PASSED")
print(f"Rows: {len(e008_submission)}")
print(f"Unique ids: {e008_submission[ID_COLUMN].nunique()}")
print(
    f"Prediction range: [{e008_submission[TARGET_COLUMN].min():.6f}, "
    f"{e008_submission[TARGET_COLUMN].max():.6f}]"
)
print(f"Missing predictions: {e008_submission[TARGET_COLUMN].isna().sum()}")
print(f"Saved to: {out_path}")

E008 submission schema validation: PASSED
Rows: 296302
Unique ids: 296302
Prediction range: [0.000197, 0.999999]
Missing predictions: 0
Saved to: D:\Projects\kaggle-smartphone-addiction\deliverables\submission_E008_catboost_screen_residual.csv


## 6. E008 public LB reconciliation

**Public LB: pending manual Kaggle upload and score entry** (see
`docs/COMPETITION_CLOSE_CHECKLIST.md`). Once the user supplies the
actual score, it will be recorded in `experiments/experiments.csv` (E008
row) and reconciled here against E008's CV mean, following the same
gap-magnitude comparison Build 8 used for E001/E002/E004/E006/E010
(`outputs/cv_lb_reconciliation.csv`: observed LB-CV gaps ranged
0.00111-0.00209, boosting-model gaps specifically 0.00111-0.00163). No
model change will follow from this result -- evidence collection only.

In [6]:
e008_public_lb = None  # placeholder -- fill in once Kaggle returns the score

if e008_public_lb is not None:
    e008_cv_mean = 0.96104
    gap = e008_public_lb - e008_cv_mean
    print(f"E008 CV mean: {e008_cv_mean}")
    print(f"E008 public LB: {e008_public_lb}")
    print(f"LB - CV gap: {gap:.5f}")
    print("Prior boosting-model gap range (Build 8): [0.00111, 0.00163]")
else:
    print("E008 public LB not yet available -- pending manual Kaggle upload.")

E008 public LB not yet available -- pending manual Kaggle upload.


## 7. Private-LB risk matrix

Qualitative risk comparison across the two selected final candidates.

In [7]:
risk_matrix = pd.DataFrame([
    {
        "candidate": "E010",
        "cv_strength": "Strongest (0.96499)",
        "cv_stability": "Strong (std 0.00051, best of the XGBoost lineage)",
        "public_lb_evidence": "Confirmed (0.96653)",
        "prediction_diversity": "N/A (reference model)",
        "model_family_diversity": "None alone -- same family as E004/E006",
        "feature_dependence": "raw + screen_residual",
        "public_lb_selection_bias_exposure": "Low (Build 8 audit)",
        "private_lb_robustness_role": "Best individual model; carries most of the expected score",
        "overall_recommendation": "Primary",
    },
    {
        "candidate": "E008",
        "cv_strength": "Moderate (0.96104, ~0.004 below E010)",
        "cv_stability": "Strong (std 0.00055, in the same tight band as other boosters)",
        "public_lb_evidence": "Pending (submitted this build)",
        "prediction_diversity": "High vs E010 (0.985 Pearson, 28.1% top-decile disagreement)",
        "model_family_diversity": "CatBoost -- only cross-family hedge available",
        "feature_dependence": "raw + screen_residual (same as E010)",
        "public_lb_selection_bias_exposure": "Low (decision made before observing LB)",
        "private_lb_robustness_role": "Diversification hedge against XGBoost-specific private-LB risk",
        "overall_recommendation": "Hedge (second final submission)",
    },
])
risk_matrix

,candidate,cv_strength,cv_stability,public_lb_evidence,prediction_diversity,model_family_diversity,feature_dependence,public_lb_selection_bias_exposure,private_lb_robustness_role,overall_recommendation
0,E010,Strongest (0.96499),"Strong (std 0.00051, best of the XGBoost lineage)",Confirmed (0.96653),N/A (reference model),None alone -- same family as E004/E006,raw + screen_residual,Low (Build 8 audit),Best individual model; carries most of the exp...,Primary
1,E008,"Moderate (0.96104, ~0.004 below E010)","Strong (std 0.00055, in the same tight band as...",Pending (submitted this build),"High vs E010 (0.985 Pearson, 28.1% top-decile ...",CatBoost -- only cross-family hedge available,raw + screen_residual (same as E010),Low (decision made before observing LB),Diversification hedge against XGBoost-specific...,Hedge (second final submission)


## 8. Final candidate decision

**Primary final candidate:** E010 (XGBoost tuned + `screen_residual`) --
CV 0.96499, public LB 0.96653.

**Second final candidate:** E008 (CatBoost + `screen_residual`) -- CV
0.96104, public LB pending. Selected for genuine model-family and
prediction diversity, not as a "second-highest public LB" default (it
had no public LB at selection time).

Not a private-LB prediction -- a deliberate hedge against the
model-family-concentration risk Build 8 identified as E010's primary
weakness.

## 9. Submission artifact audit

Both selected candidates' submission files are validated against
`data/sample_submission.csv`'s exact schema and id order.

In [8]:
audit_rows = []
for exp_id, fname in [
    ("E010", "submission_E010_xgb_tuned.csv"),
    ("E008", "submission_E008_catboost_screen_residual.csv"),
]:
    path = DELIVERABLES_DIR / fname
    sub = pd.read_csv(path)
    validate_submission(sub, sample)
    audit_rows.append({
        "experiment_id": exp_id,
        "filename": fname,
        "rows": len(sub),
        "id_validation": "PASSED (exact match to sample_submission id order)",
        "prediction_validation": "PASSED ([0,1], no missing, no non-finite)",
        "provenance": "outputs/test_predictions/{}.csv (cv_mean-verified)".format(exp_id),
        "ready_for_kaggle_selection": True,
    })

pd.DataFrame(audit_rows)

,experiment_id,filename,rows,id_validation,prediction_validation,provenance,ready_for_kaggle_selection
0,E010,submission_E010_xgb_tuned.csv,296302,PASSED (exact match to sample_submission id or...,"PASSED ([0,1], no missing, no non-finite)",outputs/test_predictions/E010.csv (cv_mean-ver...,True
1,E008,submission_E008_catboost_screen_residual.csv,296302,PASSED (exact match to sample_submission id or...,"PASSED ([0,1], no missing, no non-finite)",outputs/test_predictions/E008.csv (cv_mean-ver...,True


## 10. Competition-close actions

See `docs/COMPETITION_CLOSE_CHECKLIST.md` for the full human checklist:
upload both files (if not already uploaded), record E008's public LB,
mark exactly E010 and E008 as the two Final Submissions before the
2026-08-31 23:59 UTC deadline, and verify the marked scores match.

## 11. Build 9 conclusion

- **Primary final candidate:** E010 -- confirmed, integrity audited, no
  changes.
- **Second final candidate:** E008 -- selected as a deliberate
  model-family diversity hedge, submitted with its frozen configuration,
  decision recorded before observing its public LB.
- **No new modeling performed.** No features, tuning, blending, or
  stacking were introduced this build.
- **Further modeling before competition close: Frozen**, pending no
  integrity defect being found.
- **Pending:** E008 public LB score, private LB, final rank, final
  percentile, medal/status -- all explicitly marked pending until Kaggle
  publishes them.
- **Next build:** Build C - Consolidation, only after final judging
  selections are confirmed and (ideally) competition results are
  available.